# All-turbine selected icing-alarm forecasts: 6-hour purged temporal cross-validation

**Goal:** use the preceding **one hour** of sensor readings to forecast a **new selected icing episode (8230, 8234, 8235 or 8236) in the next six hours**, issuing a prediction every ten minutes. All turbines with eligible sensor observations are included. Site-level alarm-only IDs (for example 91) are audited but cannot supply training examples.

**Development is restricted to 2016–2023.** This notebook does not load, inspect, preprocess or score 2024–2025 unless the explicitly disabled reserved-test cell is enabled later. Those years have already been viewed in earlier project experiments, so they are reserved prospectively here, not historically pristine holdouts. Missing 2019 data is reported rather than fabricated.

The target includes **only 8230, 8234, 8235 and 8236**. Code 8210 and all other alarm codes are excluded from target activity, episode merging and labels. All turbines still participate; this changes the target code set, not the turbine selection. All original event records remain unchanged. Exact duplicate target events are removed in this experiment's derived cache, and overlapping activity is combined into episodes. This targets a new episode, not a transition between codes during an existing episode. Unknown/zero durations use start-bin evidence only. No alarm fields, turbine IDs or timestamp values enter the feature matrix.

**Run order:** processed input from `03_preprocessing` → settings → development audit/cache → CV → development plots → final refit/freeze. The reserved-test section remains off during Run All. The helper `cv_pipeline.py` contains the implementation; no changes to earlier training notebooks are required.

Changing ALARM_CODES changes both the feature-cache signature and experiment run ID. Rerun development and final refitting for this restricted target; historical broader-family results are preserved and do not describe this run.


In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'notebooks' / '01_data_export' / '01_export_yearly_turbine_scada.ipynb').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run from Iteration2 or a folder within its notebooks directory.')
MODULE_FOLDER = ROOT / 'notebooks' / '06_all_turbine_cross_validation'
if str(MODULE_FOLDER) not in sys.path:
    sys.path.insert(0, str(MODULE_FOLDER))
import cv_pipeline as cv
from sklearn.metrics import precision_recall_curve, ConfusionMatrixDisplay
import joblib

ALARM_CODES = ['8230', '8234', '8235', '8236']
# Only these four codes contribute to target episodes; 8210 and other 8200-family codes are excluded.
DEVELOPMENT_YEARS = list(range(2016, 2024))
RESERVED_YEARS = [2024, 2025]
TURBINE_IDS = None  # None = every available sensor turbine. Optional list for a smaller pilot.
SELECTED_SENSORS = None  # None = all development sensor columns, excluding alarm metadata.
HORIZON = cv.HORIZON       # Six hours (fixed for this notebook).
LOOKBACK = cv.LOOKBACK     # One hour; six prior ten-minute observations.
SEED = 42
MAX_TRAIN_NEGATIVES = 250_000  # Per fold/refit; None uses ALL negatives, requiring more RAM/time.
THRESHOLD_POLICY = 'max_f1'  # Or 'false_warning_budget': maximise event recall under constraints below.
FALSE_WARNING_BUDGET = 1.0   # False warning episodes per 30 eligible turbine-days (calibration only).
MAX_WARNING_FRACTION = 0.05  # Budget policy also limits warning time, preventing always-on warnings.
REBUILD_FEATURE_CACHE = False
RUN_CROSS_VALIDATION = True
RUN_FINAL_REFIT = True
RUN_RESERVED_TEST = False    # KEEP FALSE during development; reserved evaluation is at the bottom.
ALLOW_REPEAT_RESERVED_TEST = False
SOURCE_ROOT = ROOT / 'Data_Parquets_10min'
CACHE_ROOT = ROOT / 'Data_Parquets_CV' / 'alarms_8200_family_6hour'

# Each fold separates model fitting, threshold calibration, and outer evaluation.
FOLDS = [
 {'name': 'evaluate_2021', 'train_begin': '2016-01-01', 'calibration_begin': '2020-01-01',
  'evaluation_begin': '2021-01-01', 'end': '2022-01-01'},
 {'name': 'evaluate_2022', 'train_begin': '2016-01-01', 'calibration_begin': '2021-01-01',
  'evaluation_begin': '2022-01-01', 'end': '2023-01-01'},
 {'name': 'evaluate_2023', 'train_begin': '2016-01-01', 'calibration_begin': '2022-01-01',
  'evaluation_begin': '2023-01-01', 'end': '2024-01-01'},
]
# A small predeclared parameter search; do not change it after opening reserved results.
CANDIDATES = [
 {'name': 'constant_prevalence', 'kind': 'constant'},
 {'name': 'random_weight', 'kind': 'random'},
 {'name': 'logreg_C01', 'kind': 'logistic', 'parameters': {'C': 0.1}},
 {'name': 'logreg_C1', 'kind': 'logistic', 'parameters': {'C': 1.0}},
 {'name': 'lightgbm_leaves15', 'kind': 'lightgbm', 'parameters': {
  'n_estimators': 200, 'learning_rate': .05, 'num_leaves': 15, 'max_depth': 6}},
 {'name': 'lightgbm_leaves31', 'kind': 'lightgbm', 'parameters': {
  'n_estimators': 200, 'learning_rate': .05, 'num_leaves': 31, 'max_depth': 6}},
 {'name': 'forest_depth8', 'kind': 'forest', 'parameters': {
  'n_estimators': 100, 'max_depth': 8, 'min_samples_leaf': 10}},
 {'name': 'forest_depth12', 'kind': 'forest', 'parameters': {
  'n_estimators': 100, 'max_depth': 12, 'min_samples_leaf': 10}},
]
assert set(DEVELOPMENT_YEARS).isdisjoint(RESERVED_YEARS)
assert all(2016 <= year <= 2023 for year in DEVELOPMENT_YEARS)
assert THRESHOLD_POLICY in ('max_f1', 'false_warning_budget')
display(pd.DataFrame(FOLDS))


## Fold meaning and leakage protection

| Outer fold | Fit model/preprocessing | Choose threshold | Evaluate |
|---|---|---|---|
| 1 | Available 2016–2019 (currently 2016–2018) | 2020 | 2021 |
| 2 | Available 2016–2020 | 2021 | 2022 |
| 3 | Available 2016–2021 | 2022 | 2023 |

The calibration year is not used to fit that fold's model. Outer evaluation labels do not choose that fold's threshold. Candidate selection uses the outer development results; it is not a final unbiased performance claim after selection. Date boundaries are shared across all turbines. Input histories must stay within a split and future labels must finish before its end. For simplicity, every turbine-year cache also excludes its first input hour and last six forecast hours, even between adjacent available years.

Sensor statistics are computed separately per turbine-year using a one-step shift. Missing historical observations invalidate the window; missing individual signals remain NaN until fold-specific imputation. Median imputation, constant-feature removal and scaling are fitted on that fold's **sampled training rows only**. All positive windows are retained; the optional negative sample represents every nonempty turbine-year stratum. Estimator weights account for sampling and balance the full training populations. Preprocessor medians themselves are not inverse-probability weighted. Calibration/evaluation are never downsampled.

The default threshold maximises calibration F1. The optional warning-budget policy selects among a fixed score-quantile grid, limits false-warning episodes and warning fraction, and maximises event recall. A calibration budget is not a promise that later evaluation will meet it.


In [ ]:
# DEVELOPMENT ONLY: this enumerates and opens no 2024/2025 file.
source_paths, absent_years = cv.discover(SOURCE_ROOT, DEVELOPMENT_YEARS)
if TURBINE_IDS is not None:
    wanted = {str(t) for t in TURBINE_IDS}
    source_paths = [p for p in source_paths if p.parent.name.removeprefix('Station_') in wanted]
if not source_paths:
    raise FileNotFoundError('No development files; run the preprocessing notebook first.')
print('Unavailable development years:', absent_years)
print('Development turbine-years:', len(source_paths))
print('Candidate turbine IDs:', sorted({p.parent.name.removeprefix('Station_') for p in source_paths}))
available_sensors = cv.sensor_names(source_paths)
sensors = available_sensors if SELECTED_SENSORS is None else list(SELECTED_SENSORS)
if not sensors or len(set(sensors)) != len(sensors) or set(sensors) - set(available_sensors):
    raise ValueError('Choose distinct existing wtc_ sensors, never target/metadata fields.')
if any(not name.startswith('wtc_') for name in sensors):
    raise ValueError('Only sensor columns may be model inputs.')
feature_names = [f'{sensor}__{stat}' for stat in cv.FEATURE_STATS for sensor in sensors]
print(f'{len(sensors)} sensors -> {len(feature_names)} rolling model inputs')

config = {'alarm_codes': ALARM_CODES, 'sensors': sensors, 'development_years': DEVELOPMENT_YEARS,
          'reserved_years': RESERVED_YEARS, 'turbine_selection': TURBINE_IDS, 'seed': SEED,
          'max_train_negatives': MAX_TRAIN_NEGATIVES, 'threshold_policy': THRESHOLD_POLICY,
          'false_warning_budget': FALSE_WARNING_BUDGET, 'max_warning_fraction': MAX_WARNING_FRACTION,
          'horizon': str(HORIZON), 'lookback': str(LOOKBACK), 'folds': FOLDS, 'candidates': CANDIDATES,
          'source_signatures': [{'path': str(p.resolve()), 'size': p.stat().st_size,
                                 'mtime_ns': p.stat().st_mtime_ns} for p in source_paths],
          'implementation_sha256': hashlib.sha256((MODULE_FOLDER / 'cv_pipeline.py').read_bytes()).hexdigest()}
RUN_ID = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]
OUTPUT = ROOT / 'reports' / 'cross_validation_8200_family_6hour' / RUN_ID
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'development_protocol.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
print('Experiment output:', OUTPUT)


In [ ]:
# Caches are prepared one turbine-year at a time to limit memory.
entries = cv.prepare(source_paths, CACHE_ROOT / 'development', ALARM_CODES, sensors, REBUILD_FEATURE_CACHE)
audit = pd.DataFrame([{k: e[k] for k in ['year', 'turbine', 'observed_sensor_rows', 'eligible_rows',
    'positive_windows', 'exact_duplicates_removed', 'unknown_or_zero_durations', 'target_episode_count']}
    for e in entries])
audit.to_csv(OUTPUT / 'development_data_audit.csv', index=False)
display(audit.groupby('year').agg(turbine_ids=('turbine', 'nunique'),
    observed_rows=('observed_sensor_rows', 'sum'), eligible_windows=('eligible_rows', 'sum'),
    positive_windows=('positive_windows', 'sum'), duplicates_removed=('exact_duplicates_removed', 'sum')))
active_turbines = sorted(audit.loc[audit.eligible_rows.gt(0), 'turbine'].unique())
print('Turbines with usable prediction histories:', active_turbines)
print('Excluded turbine-years (no usable history):')
display(audit.loc[audit.eligible_rows.eq(0)])
code_records = []
for e in entries:
    for code, count in e['code_records'].items():
        code_records.append({'year': e['year'], 'turbine': e['turbine'], 'code': code, 'starts': count})
code_audit = pd.DataFrame(code_records, columns=['year', 'turbine', 'code', 'starts'])
code_audit.to_csv(OUTPUT / 'development_code_counts.csv', index=False)
if not code_audit.empty:
    display(code_audit.groupby('code').starts.sum().rename('Deduplicated development starts').to_frame())
print('Deduplication affects derived CV caches only. Original Parquet files are unchanged.')


In [ ]:
# Runs 3 temporal folds × 8 candidate configurations; can take substantial CPU time.
if RUN_CROSS_VALIDATION:
    cv_metrics, cv_summary = cv.run_cv(entries, feature_names, CANDIDATES, FOLDS, OUTPUT / 'cv',
        max_negatives=MAX_TRAIN_NEGATIVES, seed=SEED, threshold_policy=THRESHOLD_POLICY,
        false_warning_budget=FALSE_WARNING_BUDGET, max_warning_fraction=MAX_WARNING_FRACTION)
else:
    cv_metrics = pd.read_csv(OUTPUT / 'cv' / 'cv_metrics.csv')
    cv_summary = pd.read_csv(OUTPUT / 'cv' / 'cv_summary.csv', index_col='candidate')
display(cv_summary)
display(cv_metrics[['fold', 'candidate', 'AP', 'macro_AP', 'prevalence', 'precision', 'recall',
    'F1', 'F2', 'TP', 'FP', 'FN', 'eligible_alarm_episodes', 'event_recall',
    'false_warning_episodes_per_30_eligible_turbine_days', 'median_detected_lead_hours']])
print('AP is average precision. Compare it with the prevalence in each fold.')
print('macro_AP averages turbines with both classes; the count with defined AP is in cv_metrics.csv.')


In [ ]:
# Fold and per-turbine development comparisons; no reserved data is involved.
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
cv_metrics.pivot(index='fold', columns='candidate', values='AP').plot.bar(ax=axes[0])
axes[0].set(title='Outer-fold average precision', ylabel='AP'); axes[0].legend(fontsize=7)
cv_metrics.pivot(index='fold', columns='candidate', values='event_recall').plot.bar(ax=axes[1])
axes[1].set(title='Eligible alarm-episode recall', ylabel='Event recall'); axes[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUTPUT / 'cv_fold_comparison.png', dpi=150); plt.show()
per_turbine = pd.read_csv(OUTPUT / 'cv' / 'cv_per_turbine.csv')
macro = per_turbine.groupby(['candidate', 'turbine']).AP.mean().unstack('candidate')
display(macro)
fig, ax = plt.subplots(figsize=(12, max(5, len(macro) * .3)))
image = ax.imshow(macro.to_numpy(), aspect='auto')
ax.set(yticks=range(len(macro)), yticklabels=macro.index,
       xticks=range(len(macro.columns)), xticklabels=macro.columns, title='Mean outer-fold AP per turbine')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right'); fig.colorbar(image, ax=ax, label='AP')
fig.tight_layout(); fig.savefig(OUTPUT / 'cv_per_turbine_AP.png', dpi=150); plt.show()


In [ ]:
# Confusion matrices and PR curves use independently calibrated outer-fold decisions.
fig, axes = plt.subplots(1, len(FOLDS), figsize=(6 * len(FOLDS), 5), squeeze=False)
for ax, fold in zip(axes[0], FOLDS):
    for candidate in CANDIDATES:
        prediction = pd.read_parquet(OUTPUT / 'cv' / 'oof' / candidate['name'] / f"{fold['name']}.parquet")
        p, r, _ = precision_recall_curve(prediction.target, prediction.score)
        ax.plot(r, p, label=candidate['name'])
    ax.axhline(prediction.target.mean(), color='grey', linestyle='--', label='Prevalence')
    ax.set(title=fold['name'], xlabel='Recall', ylabel='Precision'); ax.legend(fontsize=7)
fig.suptitle('Icing 8230/8234/8235/8236 | preceding hour -> next 6 hours | development only')
fig.tight_layout(); fig.savefig(OUTPUT / 'cv_precision_recall.png', dpi=150); plt.show()
# Best development candidate per learned family, plus the two baselines.
representatives = []
for kind in dict.fromkeys(c['kind'] for c in CANDIDATES):
    family = [c['name'] for c in CANDIDATES if c['kind'] == kind]
    representatives.append(cv_summary.loc[family, 'mean_AP'].idxmax())
fig, axes = plt.subplots(len(representatives), len(FOLDS), figsize=(5 * len(FOLDS), 4 * len(representatives)), squeeze=False)
for row, name in enumerate(representatives):
    for col, fold in enumerate(FOLDS):
        prediction = pd.read_parquet(OUTPUT / 'cv' / 'oof' / name / f"{fold['name']}.parquet")
        threshold = cv_metrics.loc[cv_metrics.candidate.eq(name) & cv_metrics.fold.eq(fold['name']), 'threshold'].iloc[0]
        ConfusionMatrixDisplay.from_predictions(prediction.target, prediction.score.ge(threshold),
            labels=[0, 1], display_labels=['No new episode', 'Selected icing onset'],
            ax=axes[row, col], colorbar=False, values_format='d')
        axes[row, col].set_title(f"{name} | {fold['name']}")
fig.tight_layout(); fig.savefig(OUTPUT / 'cv_confusion_matrices.png', dpi=150); plt.show()


## Final development refit and frozen test protocol

The highest mean outer-fold AP chooses the parameter setting within each model family and identifies a learned-model champion. Baselines are retained as comparisons. Final thresholds are selected using the chosen candidates' 2021–2023 out-of-fold development predictions. All final models/preprocessors are then fitted on available 2016–2023 data, with the same documented negative-sampling policy. Their thresholds, code list, sensor list, source signatures and configuration are saved before test access.

Final refitting may change score distributions, so a threshold or warning budget that worked on development predictions is not guaranteed to hold later. Reserved testing measures this frozen procedure; it must not retune thresholds. The fold tables above assess the fold-specific inner-calibrated procedures, not an independently evaluated final-refit threshold.


In [ ]:
if RUN_FINAL_REFIT:
    final_bundle = cv.final_refit(entries, feature_names, CANDIDATES, FOLDS,
                                 OUTPUT / 'cv', OUTPUT / 'final', config)
    print('Frozen champion:', final_bundle['champion'])
    display(pd.Series(final_bundle['thresholds'], name='Frozen development threshold').to_frame())
else:
    print('Final refit skipped. Enable RUN_FINAL_REFIT after reviewing the development results.')
print('Final model location:', OUTPUT / 'final' / 'frozen_models.joblib')


## Reserved test: intentionally off

Keep **RUN_RESERVED_TEST = False** while selecting features, parameters, targets or thresholds. With it false, this section does not even discover reserved source files. Only when the development configuration is final should you set it true and rerun the cell below. It then loads 2024–2025 for turbines represented in final training, applies frozen preprocessing/weights/thresholds, and saves pooled/per-turbine results and predictions. It does not assess generalisation to unseen turbines.

Existing test results cannot be silently overwritten: an intentional repeat requires ALLOW_REPEAT_RESERVED_TEST. A previous project has already inspected these years, so report that historical limitation even if this notebook has not accessed them. No interval of these reserved years should be used to choose settings for this experiment.


In [ ]:
if not RUN_RESERVED_TEST:
    reserved_results = cv.reserved_test({}, SOURCE_ROOT, CACHE_ROOT / 'reserved', OUTPUT / 'reserved_test', enabled=False)
else:
    frozen_path = OUTPUT / 'final' / 'frozen_models.joblib'
    if not frozen_path.is_file():
        raise FileNotFoundError('Finish and freeze the development refit before enabling reserved testing.')
    frozen = joblib.load(frozen_path)
    reserved_results = cv.reserved_test(frozen, SOURCE_ROOT, CACHE_ROOT / 'reserved', OUTPUT / 'reserved_test',
                                       enabled=True, allow_repeat=ALLOW_REPEAT_RESERVED_TEST)
    display(reserved_results)


In [ ]:
# Reserved plots are generated only after explicit, completed reserved evaluation.
if RUN_RESERVED_TEST and reserved_results is not None:
    fig, ax = plt.subplots(figsize=(9, 5))
    names = reserved_results.candidate.tolist()
    for name in names:
        prediction = pd.read_parquet(OUTPUT / 'reserved_test' / f'predictions_{name}.parquet')
        p, r, _ = precision_recall_curve(prediction.target, prediction.score)
        ax.plot(r, p, label=name)
    ax.axhline(prediction.target.mean(), color='grey', linestyle='--', label='Reserved prevalence')
    ax.set(title='Frozen selected-icing models: reserved 2024–2025', xlabel='Recall', ylabel='Precision')
    ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(OUTPUT / 'reserved_test' / 'precision_recall.png', dpi=150); plt.show()
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, name in zip(axes.flat, names):
        prediction = pd.read_parquet(OUTPUT / 'reserved_test' / f'predictions_{name}.parquet')
        ConfusionMatrixDisplay.from_predictions(prediction.target, prediction.score.ge(frozen['thresholds'][name]),
            labels=[0, 1], display_labels=['No new episode', 'Selected icing onset'], ax=ax, colorbar=False, values_format='d')
        ax.set_title(name)
    for ax in axes.flat[len(names):]:
        ax.axis('off')
    fig.tight_layout(); fig.savefig(OUTPUT / 'reserved_test' / 'confusion_matrices.png', dpi=150); plt.show()
else:
    print('No reserved-test plots: 2024–2025 remain unopened by this workflow.')


## Interpretation and saved files

- `development_data_audit.csv` reports eligible sensor windows, positive windows and duplicate handling by turbine-year. `development_code_counts.csv` identifies actual code availability.
- `cv/cv_metrics.csv`, `cv/cv_summary.csv` and `cv/cv_per_turbine.csv` report pooled and turbine-level outer development results. Fold standard deviations describe temporal variation; they are not independent-event confidence intervals.
- `cv/oof/` contains outer-fold scores only. `cv/sampling_audit.json` states which turbines/rows were represented and the unthinned class prevalence.
- `final/frozen_models.joblib` and `final/frozen_configuration.json` record the final development-only models and thresholds. Do not load model files from untrusted sources.
- `reserved_test/` is created only after explicitly enabling held-out evaluation.

**AP** is average precision; compare with prevalence. **Event recall** counts merged target episodes with at least one eligible prediction opportunity in the preceding six hours, detected by at least one warning. **Lead time** is measured from the earliest qualifying warning to episode onset. Consecutive positive decisions ten minutes apart merge into warning episodes; false episodes have no target onset in their covered forecast intervals. False-warning rates use 30 days of eligible prediction-time exposure, not full calendar time. The optional budget also constrains warning fraction to avoid rewarding permanent warnings.

Alarm records may be incomplete; missing end times have uncertain activity duration. No amount of negative sampling or cross-validation invents independent positive events. Scores from class-balanced fitting and the random baseline are not calibrated real-world probabilities. Reserve separate turbines as well if the research question changes to transfer to machines never seen during training.
